In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

khuynhongvn_neo4j_dump_path = kagglehub.dataset_download('khuynhongvn/neo4j-dump')
khuynhongvn_neo4j_plugins_path = kagglehub.dataset_download('khuynhongvn/neo4j-plugins')
khuynhongvn_neo4j_dump2_path = kagglehub.dataset_download('khuynhongvn/neo4j-dump2')

print('Data source import complete.')


In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

khuynhongvn_neo4j_plugins_path = kagglehub.dataset_download('khuynhongvn/neo4j-plugins')
khuynhongvn_neo4j_dump2_path = kagglehub.dataset_download('khuynhongvn/neo4j-dump2')

print('Data source import complete.')


In [ ]:
import kagglehub
kagglehub.login()


In [ ]:
khuynhongvn_neo4j_plugins_path = kagglehub.dataset_download('khuynhongvn/neo4j-plugins')
khuynhongvn_neo4j_dump2_path = kagglehub.dataset_download('khuynhongvn/neo4j-dump2')

print('Data source import complete.')


In [ ]:
import kagglehub
kagglehub.login()


In [ ]:
khuynhongvn_neo4j_plugins_path = kagglehub.dataset_download('khuynhongvn/neo4j-plugins')
khuynhongvn_neo4j_dump2_path = kagglehub.dataset_download('khuynhongvn/neo4j-dump2')

print('Data source import complete.')


# AML Trust Scoring Pipeline

## Bước 0 — Cài đặt Neo4j & Java

In [ ]:
# Kaggle data import — chỉ chạy 1 lần
import kagglehub
kagglehub.login()

khuynhongvn_neo4j_dump_path    = kagglehub.dataset_download('khuynhongvn/neo4j-dump')
khuynhongvn_neo4j_plugins_path = kagglehub.dataset_download('khuynhongvn/neo4j-plugins')
khuynhongvn_neo4j_dump2_path   = kagglehub.dataset_download('khuynhongvn/neo4j-dump2')
print('Data source import complete.')

In [ ]:
!wget -q https://dist.neo4j.org/neo4j-community-2026.03.1-unix.tar.gz
!tar -xzf neo4j-community-2026.03.1-unix.tar.gz

In [ ]:
!apt-get update -qq
!apt-get install -y -qq openjdk-21-jdk

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libatspi2.0-0:amd64.
(Reading database ... 120314 files and directories currently installed.)
Preparing to unpack .../00-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../01-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../02-session-migration_0.3.6_amd64.deb ...
Unpacking session-migration (0.3.6) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../03-gsettings-desktop-schemas_42.0-1ubuntu1_all.deb ...
Unpacking gsettings-desktop-schemas (42.0-1ubuntu1) ...
Selecting previously unselected

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
!java -version

openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)


In [ ]:
!neo4j-community-2026.03.1/bin/neo4j-admin database load neo4j \
  --from-path=/kaggle/input/datasets/khuynhongvn/neo4j-dump2 \
  --overwrite-destination=true

Done: 38 files, 5.561GiB processed in 33.658 seconds.


In [ ]:
import shutil
src = "/kaggle/input/datasets/khuynhongvn/neo4j-plugins/neo4j-graph-data-science-2.27.0.jar"
dst = "/kaggle/working/neo4j-community-2026.03.1/plugins/neo4j-graph-data-science.jar"
shutil.copy(src, dst)
print("✓ GDS plugin copied.")

✓ GDS plugin copied.


In [ ]:
config_path = "/kaggle/working/neo4j-community-2026.03.1/conf/neo4j.conf"

with open(config_path, "r") as f:
    lines = f.readlines()

new_lines = []
has_plugins_line = False
has_unrestricted = False
has_allowlist = False

for line in lines:
    if line.strip().startswith("#server.directories.plugins"):
        new_lines.append("server.directories.plugins=plugins\n")
        has_plugins_line = True
    else:
        new_lines.append(line)
    if "server.directories.plugins=plugins" in line:
        has_plugins_line = True
    if "dbms.security.procedures.unrestricted" in line:
        has_unrestricted = True
    if "dbms.security.procedures.allowlist" in line:
        has_allowlist = True

if not has_plugins_line:
    new_lines.append("\nserver.directories.plugins=plugins\n")
if not has_unrestricted:
    new_lines.append("dbms.security.procedures.unrestricted=gds.*,apoc.*\n")
if not has_allowlist:
    new_lines.append("dbms.security.procedures.allowlist=gds.*,apoc.*\n")

with open(config_path, "w") as f:
    f.writelines(new_lines)

print("✓ Đã cấu hình Neo4j config xong")

✓ Đã cấu hình Neo4j config xong


In [ ]:
!sed -i 's/#dbms.default_listen_address=0.0.0.0/dbms.default_listen_address=127.0.0.1/' neo4j-community-2026.03.1/conf/neo4j.conf
!echo "dbms.security.auth_enabled=false" >> neo4j-community-2026.03.1/conf/neo4j.conf
# !echo "server.bolt.advertised_address=127.0.0.1:7687" >> /kaggle/working/neo4j-community-2026.03.1/conf/neo4j.conf

In [ ]:
!./neo4j-community-2026.03.1/bin/neo4j console > /kaggle/working/neo4j.log 2>&1

## Bước 1 — Cài đặt thư viện Python

In [1]:
!pip install -q neo4j scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 16.4 MB/s eta 0:00:00


## Bước 2 — Cấu hình hệ thống

Tập trung toàn bộ tham số vào một chỗ.

In [2]:
import os

# ── Kết nối Neo4j ──────────────────────────────────────────────────
NEO4J_URI      = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")

# ── GDS Graph projection ───────────────────────────────────────────
GRAPH_NAME        = "aml_graph"
NODE_LABEL        = "Account"
RELATIONSHIP_TYPE = "TRANSFER"

# ── PageRank ──────────────────────────────────────────────────────
PAGERANK_CONFIG = {
    "maxIterations": 300,
    "dampingFactor": 0.85,
    "tolerance":     1e-7,
    "useWeights":    True,
}

# ── TrustRank ─────────────────────────────────────────────────────
TRUSTRANK_CONFIG = {
    "maxIterations": 100,
    "dampingFactor": 0.85,
    "tolerance":     1e-6,
    "seed_label":    0,
    "max_seed_count": 5000,
    "seed_strategy": "random",
}

# ── Anti-TrustRank ────────────────────────────────────────────────
ANTI_TRUSTRANK_CONFIG = {
    "maxIterations": 100,
    "dampingFactor": 0.85,
    "tolerance":     1e-7,
    "seed_label":    1,
    "max_seed_count": 5000,
    "seed_strategy": "random",
}

TRUST_SCORE_WEIGHTS = {
    "trustrank_w":      0.00,
    "pagerank_w":       0.20,
    "anti_trustrank_w": 0.80,
}

# ── Credit score mapping ──────────────────────────────────────────
CREDIT_SCORE_MIN = 300
CREDIT_SCORE_MAX = 900

print("✓ Config loaded.")
print(f"  Weights: {TRUST_SCORE_WEIGHTS}")
print(f"  TrustRank seeds: {TRUSTRANK_CONFIG['max_seed_count']} ({TRUSTRANK_CONFIG['seed_strategy']})")

✓ Config loaded.
  Weights: {'trustrank_w': 0.0, 'pagerank_w': 0.2, 'anti_trustrank_w': 0.8}
  TrustRank seeds: 5000 (random)


## Bước 3 — Kết nối Neo4j

In [3]:
from neo4j import GraphDatabase

class Neo4jConnector:

    def __init__(self, uri=NEO4J_URI, user=NEO4J_USER, password=NEO4J_PASSWORD):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def run(self, query: str, params: dict = None):
        params = params or {}
        with self.driver.session() as session:
            result = session.run(query, **params)
            return [record.data() for record in result]

    def run_write(self, query: str, params: dict = None):
        params = params or {}
        with self.driver.session() as session:
            return session.execute_write(
                lambda tx: tx.run(query, **params).consume()
            )

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()


conn = Neo4jConnector()

try:
    result = conn.run("RETURN 1 AS ping")
    print(f"✓ Neo4j connected: {result}")
except Exception as e:
    print(f"✗ Kết nối thất bại: {e}")

✓ Neo4j connected: [{'ping': 1}]


## Bước 4 — Cập nhật label từ giao dịch gian lận

Gán `label=1` cho các account tham gia giao dịch `is_laundering=1`.

In [4]:
import time

BATCH_SIZE = 10000

def update_account_labels(conn: Neo4jConnector):

    update_query = """
        MATCH (src:Account)-[r:TRANSFER]->(dst:Account)
        WHERE r.is_laundering = 1
          AND id(r) > $last_rel_id
        WITH src, dst, r
        ORDER BY id(r)
        LIMIT $batch_size
        SET src.label = 1,
            dst.label = 1
        RETURN count(r) AS processed,
               coalesce(max(id(r)), $last_rel_id) AS next_last_rel_id
    """
    # Gán label=0 cho account chưa có label (chưa tham gia fraud)
    set_normal_query = """
        MATCH (a:Account)
        WHERE a.label IS NULL
        SET a.label = 0
        RETURN count(a) AS updated
    """
    total_updated = 0
    last_rel_id = -1
    t0 = time.time()
    while True:
        record = conn.run(update_query, {"batch_size": BATCH_SIZE, "last_rel_id": last_rel_id})
        updated = record[0]["processed"] if record else 0
        if updated == 0:
            break
        total_updated += updated
        last_rel_id = record[0]["next_last_rel_id"]
        print(f"  Processed {total_updated} relationships | elapsed={time.time()-t0:.1f}s")

    print(f"✓ Fraud label done. Total: {total_updated} relationships.")

    # Gán 0 cho normal
    r2 = conn.run(set_normal_query)
    normal_set = r2[0]["updated"] if r2 else 0
    print(f"✓ Normal label done. Set label=0 cho {normal_set:,} accounts.")


update_account_labels(conn)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=4, column=15, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 4, 'column': 15}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (src:Account)-[r:TRANSFER]->(dst:Account)\n        WHERE r.is_laundering = 1\n          AND id(r) > $last_rel_id\n        WITH src, dst, r\n        ORDER BY id(r)\n        LIMIT $batch_size\n        SET src.label = 1,\n            dst.label = 1\n        RETURN count(r) AS processed,\n           

  Processed 10000 relationships | elapsed=9.8s


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=4, column=15, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 4, 'column': 15}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (src:Account)-[r:TRANSFER]->(dst:Account)\n        WHERE r.is_laundering = 1\n          AND id(r) > $last_rel_id\n        WITH src, dst, r\n        ORDER BY id(r)\n        LIMIT $batch_size\n        SET src.label = 1,\n            dst.label = 1\n        RETURN count(r) AS processed,\n           

  Processed 20000 relationships | elapsed=18.2s


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=4, column=15, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 4, 'column': 15}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (src:Account)-[r:TRANSFER]->(dst:Account)\n        WHERE r.is_laundering = 1\n          AND id(r) > $last_rel_id\n        WITH src, dst, r\n        ORDER BY id(r)\n        LIMIT $batch_size\n        SET src.label = 1,\n            dst.label = 1\n        RETURN count(r) AS processed,\n           

  Processed 30000 relationships | elapsed=28.5s


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=4, column=15, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 4, 'column': 15}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (src:Account)-[r:TRANSFER]->(dst:Account)\n        WHERE r.is_laundering = 1\n          AND id(r) > $last_rel_id\n        WITH src, dst, r\n        ORDER BY id(r)\n        LIMIT $batch_size\n        SET src.label = 1,\n            dst.label = 1\n        RETURN count(r) AS processed,\n           

  Processed 35230 relationships | elapsed=40.1s


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=4, column=15, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 4, 'column': 15}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (src:Account)-[r:TRANSFER]->(dst:Account)\n        WHERE r.is_laundering = 1\n          AND id(r) > $last_rel_id\n        WITH src, dst, r\n        ORDER BY id(r)\n        LIMIT $batch_size\n        SET src.label = 1,\n            dst.label = 1\n        RETURN count(r) AS processed,\n           

✓ Fraud label done. Total: 35230 relationships.
✓ Normal label done. Set label=0 cho 0 accounts.


In [5]:
query = """
MATCH (a:Account)
RETURN
    count(CASE WHEN a.label = 1 THEN 1 END) AS label_1,
    count(CASE WHEN a.label = 0 THEN 1 END) AS label_0,
    count(CASE WHEN a.label IS NULL THEN 1 END) AS label_null
"""

result = conn.run(query)[0]

print("=== Account Label Stats ===")
print(f"Label = 1   : {result['label_1']:,}")
print(f"Label = 0   : {result['label_0']:,}")
print(f"Label = NULL: {result['label_null']:,}")
if result['label_null'] > 0:
    print("⚠ Còn account chưa có label — kiểm tra lại batch size!")

=== Account Label Stats ===
Label = 1   : 41,857
Label = 0   : 2,035,166
Label = NULL: 0


## Bước 5 — Tạo GDS Graph Projection

In [6]:
import time

def drop_projection_if_exists(conn, graph_name=GRAPH_NAME):
    result = conn.run("CALL gds.graph.exists($graphName) YIELD exists RETURN exists",
                      {"graphName": graph_name})
    if result and result[0]["exists"]:
        conn.run("CALL gds.graph.drop($graphName) YIELD graphName",
                 {"graphName": graph_name})
        print(f"  Đã xóa projection cũ: '{graph_name}'")


def create_projection(conn, graph_name=GRAPH_NAME):
    drop_projection_if_exists(conn, graph_name)
    project_query = """
        CALL gds.graph.project(
            $graphName,
            { Account: { properties: ['label'] } },
            {
                TRANSFER: {
                    orientation: 'NATURAL',
                    properties: { amount: { defaultValue: 1.0 } }
                }
            }
        )
        YIELD graphName, nodeCount, relationshipCount
        RETURN graphName, nodeCount, relationshipCount
    """
    result = conn.run(project_query, {"graphName": graph_name})
    if result:
        r = result[0]
        print(f"✓ Projection '{r['graphName']}': {r['nodeCount']:,} nodes, {r['relationshipCount']:,} edges")
    return result


t0 = time.time()
create_projection(conn)
print(f"  Thời gian: {time.time()-t0:.1f}s")

✓ Projection 'aml_graph': 2,077,023 nodes, 31,898,238 edges
  Thời gian: 27.3s


## Bước 6 — PageRank

Đo tầm quan trọng tổng thể của mỗi node trong mạng giao dịch.

In [7]:
import numpy as np
import pandas as pd

def run_pagerank(conn, graph_name=GRAPH_NAME, cfg=None):
    cfg = cfg or PAGERANK_CONFIG
    use_weights = cfg.get("useWeights", False)
    print(f"[PageRank] Đang chạy... (weighted={use_weights})")

    params = {
        "graphName":     graph_name,
        "maxIterations": cfg["maxIterations"],
        "dampingFactor": cfg["dampingFactor"],
        "tolerance":     cfg["tolerance"],
    }
    write_config = ""
    if use_weights:
        write_config = "relationshipWeightProperty: 'amount',"

    write_query = f"""
        CALL gds.pageRank.write(
            $graphName,
            {{
                maxIterations:  $maxIterations,
                dampingFactor:  $dampingFactor,
                tolerance:      $tolerance,
                {write_config}
                writeProperty:  'pagerank'
            }}
        )
        YIELD nodePropertiesWritten, ranIterations, didConverge
        RETURN nodePropertiesWritten, ranIterations, didConverge
    """
    result = conn.run(write_query, params)
    if result:
        r = result[0]
        print(f"  Ghi {r['nodePropertiesWritten']:,} nodes | vòng: {r['ranIterations']} | hội tụ: {r['didConverge']}")
        if not r['didConverge']:
            print("  ⚠ PageRank chưa hội tụ — tăng maxIterations hoặc tolerance")

    records = conn.run("""
        MATCH (a:Account)
        RETURN a.id AS id, a.pagerank AS pagerank, a.label AS label
        ORDER BY pagerank DESC
    """)
    df = pd.DataFrame(records)
    if df.empty:
        print("⚠ Không đọc được kết quả PageRank!")
        return df

    print(f"✓ PageRank xong. {len(df):,} nodes.")
    print(f"  Raw: min={df['pagerank'].min():.2e}, max={df['pagerank'].max():.2e}, mean={df['pagerank'].mean():.2e}")

    # Zero-inflation check
    zero_pct = (df['pagerank'] == 0).sum() / len(df) * 100
    if zero_pct > 30:
        print(f"  ⚠ {zero_pct:.1f}% nodes có pagerank=0 (isolated nodes?)")

    if "label" in df.columns:
        g = df.groupby("label")["pagerank"].mean()
        print(f"  Avg PageRank — normal={g.get(0,0):.2e}, fraud={g.get(1,0):.2e}")
        ratio = g.get(1, 0) / (g.get(0, 1e-12))
        print(f"  Fraud/Normal ratio = {ratio:.2f}x (kỳ vọng > 1 nếu fraud là hub AML)")
    return df


t0 = time.time()
df_pr = run_pagerank(conn)
print(f"  Thời gian: {time.time()-t0:.1f}s")
df_pr.head()

[PageRank] Đang chạy... (weighted=True)
  Ghi 2,077,023 nodes | vòng: 121 | hội tụ: True
✓ PageRank xong. 2,077,023 nodes.
  Raw: min=1.50e-01, max=2.89e+02, mean=9.35e-01
  Avg PageRank — normal=9.27e-01, fraud=1.31e+00
  Fraud/Normal ratio = 1.42x (kỳ vọng > 1 nếu fraud là hub AML)
  Thời gian: 152.2s


,id,pagerank,label
0,70_100428660,289.218082,1
1,70_1004286A8,182.350654,1
2,222107_829E12A50,68.799610,0
3,2470_843786D10,67.767365,0
4,153193_8387EEB30,67.671863,0


## Bước 7 — TrustRank (Personalized PageRank từ seed sạch)

In [8]:
import random

def _get_quality_seeds(conn: Neo4jConnector, max_count: int,
                       strategy: str = "random") -> list:
    print(f"[TrustRank] Chọn tối đa {max_count:,} seed nodes (strategy={strategy})...")

    # Strict: không kề fraud
    if strategy == "random":
        strict_query = """
            MATCH (a:Account {label: 0})
            WHERE NOT EXISTS {
                MATCH (a)-[:TRANSFER]-(fraud:Account {label: 1})
            }
            RETURN a.id AS id
        """
        # Lấy toàn bộ ids rồi sample ngẫu nhiên
        all_records = conn.run(strict_query)
        all_ids = [r["id"] for r in all_records]
        if len(all_ids) >= max_count:
            ids = random.sample(all_ids, max_count)
            print(f"  Strict seed: {len(ids):,} ngẫu nhiên từ {len(all_ids):,} clean nodes")
        elif len(all_ids) >= max_count // 2:
            ids = all_ids
            print(f"  Strict seed: dùng tất cả {len(ids):,} clean nodes")
        else:
            print(f"  Strict chỉ có {len(all_ids):,}, fallback lấy label=0 ngẫu nhiên...")
            fallback_query = "MATCH (a:Account {label: 0}) RETURN a.id AS id"
            fb_records = conn.run(fallback_query)
            fb_ids = [r["id"] for r in fb_records]
            ids = random.sample(fb_ids, min(max_count, len(fb_ids)))
            print(f"  Fallback seed: {len(ids):,} ngẫu nhiên từ label=0")
    else:  # degree strategy (legacy, kept for comparison)
        strict_query = """
            MATCH (a:Account {label: 0})
            WHERE NOT EXISTS {
                MATCH (a)-[:TRANSFER]-(fraud:Account {label: 1})
            }
            WITH a, coalesce(a.in_degree, 0) + coalesce(a.out_degree, 0) AS deg
            ORDER BY deg DESC
            LIMIT $limit
            RETURN a.id AS id, deg
        """
        records = conn.run(strict_query, {"limit": max_count})
        ids = [r["id"] for r in records]
        print(f"  Degree-based seed: {len(ids):,} nodes")

    return ids


def run_trustrank(conn: Neo4jConnector, graph_name: str = GRAPH_NAME,
                  cfg: dict = None) -> pd.DataFrame:
    cfg = cfg or TRUSTRANK_CONFIG
    strategy = cfg.get("seed_strategy", "random")
    seed_ids = _get_quality_seeds(conn, cfg["max_seed_count"], strategy=strategy)

    if not seed_ids:
        raise ValueError("[TrustRank] Không có seed nodes label=0. Chạy update_account_labels() trước!")

    print(f"[TrustRank] Chạy PPR với {len(seed_ids):,} seed nodes...")
    ppr_query = """
        MATCH (a:Account)
        WHERE a.id IN $ids
        WITH collect(a) AS sources
        CALL gds.pageRank.write(
            $graphName,
            {
                maxIterations:  $maxIterations,
                dampingFactor:  $dampingFactor,
                tolerance:      $tolerance,
                sourceNodes:    sources,
                writeProperty:  'trustrank'
            }
        )
        YIELD nodePropertiesWritten, ranIterations, didConverge
        RETURN nodePropertiesWritten, ranIterations, didConverge
    """
    t0 = time.time()
    result = conn.run(ppr_query, {
        "graphName":     graph_name,
        "maxIterations": cfg["maxIterations"],
        "dampingFactor": cfg["dampingFactor"],
        "tolerance":     cfg["tolerance"],
        "ids":           seed_ids,
    })
    if result:
        r = result[0]
        print(f"  Ghi {r['nodePropertiesWritten']:,} nodes | vòng: {r['ranIterations']} | hội tụ: {r['didConverge']} | {time.time()-t0:.1f}s")
        # [AUDIT-FIX8] Cảnh báo nếu chưa hội tụ
        if not r['didConverge']:
            print("  ⚠ TrustRank CHƯA HỘI TỤ — tăng maxIterations hoặc giảm dampingFactor!")
            print("    Signal có thể không ổn định. Kiểm tra kết quả cẩn thận.")

    records = conn.run("""
        MATCH (a:Account)
        WHERE a.trustrank IS NOT NULL
        RETURN a.id AS id, a.trustrank AS trustrank, a.label AS label
        ORDER BY trustrank DESC
    """)
    if not records:
        print("⚠ Không đọc được trustrank.")
        return pd.DataFrame(columns=["id", "trustrank", "label"])

    df = pd.DataFrame(records)

    # [AUDIT-FIX9] Zero-inflation check
    zero_pct = (df['trustrank'] == 0).sum() / len(df) * 100
    median_val = df['trustrank'].median()

    print(f"\n[TrustRank] Phân bố:")
    print(f"  Min={df['trustrank'].min():.6f} | Median={median_val:.6f} | Max={df['trustrank'].max():.6f}")
    print(f"  Zero nodes: {zero_pct:.1f}%")

    if median_val == 0:
        print("  🚨 CẢNH BÁO NGHIÊM TRỌNG: Median=0 → Signal collapse!")
        print("     TrustRank không phân tán đủ. Tăng seed count hoặc giảm dampingFactor.")
    elif zero_pct > 50:
        print(f"  ⚠ {zero_pct:.1f}% nodes bằng 0 → Signal có thể diluted.")

    if "label" in df.columns:
        for lbl, name in [(0, "normal"), (1, "fraud")]:
            sub = df[df["label"] == lbl]
            if not sub.empty:
                print(f"  {name}: n={len(sub):,}, avg={sub['trustrank'].mean():.6f}, median={sub['trustrank'].median():.6f}")

        # Discriminative power check
        fraud_mean  = df[df["label"]==1]['trustrank'].mean()
        normal_mean = df[df["label"]==0]['trustrank'].mean()
        sep = normal_mean - fraud_mean
        print(f"  Separation (normal - fraud): {sep:+.6f}")
        if sep < 0:
            print("  🚨 CẢNH BÁO: Fraud có TrustRank CAO hơn Normal — seed selection sai!")
        elif abs(sep) < 1e-5:
            print("  ⚠ Separation gần 0 → TrustRank ít discriminative")
        else:
            print("  ✓ TrustRank đúng chiều: normal > fraud")

    print(f"✓ TrustRank xong. {len(df):,} nodes.")
    return df


t0 = time.time()
df_tr = run_trustrank(conn)
print(f"  Thời gian: {time.time()-t0:.1f}s")
df_tr.head()

[TrustRank] Chọn tối đa 5,000 seed nodes (strategy=random)...
  Strict seed: 5,000 ngẫu nhiên từ 1,753,422 clean nodes
[TrustRank] Chạy PPR với 5,000 seed nodes...
  Ghi 2,077,023 nodes | vòng: 81 | hội tụ: True | 14.5s

[TrustRank] Phân bố:
  Min=0.000000 | Median=0.000000 | Max=2.034012
  Zero nodes: 94.7%
  🚨 CẢNH BÁO NGHIÊM TRỌNG: Median=0 → Signal collapse!
     TrustRank không phân tán đủ. Tăng seed count hoặc giảm dampingFactor.
  normal: n=2,035,166, avg=0.002263, median=0.000000
  fraud: n=41,857, avg=0.000559, median=0.000000
  Separation (normal - fraud): +0.001704
  ✓ TrustRank đúng chiều: normal > fraud
✓ TrustRank xong. 2,077,023 nodes.
  Thời gian: 264.6s


,id,trustrank,label
0,11796_8438A1480,2.034012,0
1,98226_826557FE0,1.847809,0
2,175662_8214FC280,1.690233,0
3,258566_83CC676C0,1.589120,0
4,26967_8111D4040,1.589120,0


## Bước 8 — Anti-TrustRank (Personalized PageRank từ fraud seeds)

In [9]:
def run_anti_trustrank(conn, graph_name=GRAPH_NAME, cfg=None):
    cfg = cfg or ANTI_TRUSTRANK_CONFIG
    max_seeds = cfg["max_seed_count"]
    strategy  = cfg.get("seed_strategy", "random")
    print(f"[Anti-TrustRank] Chuẩn bị {max_seeds} fraud seed nodes (strategy={strategy})...")

    # [AUDIT-FIX4] Random seed thay vì degree
    if strategy == "random":
        all_fraud = conn.run(
            "MATCH (a:Account {label: $label}) RETURN a.id AS id",
            {"label": cfg["seed_label"]}
        )
        all_fraud_ids = [r["id"] for r in all_fraud]
        if len(all_fraud_ids) <= max_seeds:
            seed_ids = all_fraud_ids
            print(f"  Dùng tất cả {len(seed_ids):,} fraud nodes")
        else:
            seed_ids = random.sample(all_fraud_ids, max_seeds)
            print(f"  Random {len(seed_ids):,} fraud seeds từ {len(all_fraud_ids):,}")
    else:
        records = conn.run("""
            MATCH (a:Account {label: $label})
            WITH a, coalesce(a.in_degree, 0) + coalesce(a.out_degree, 0) AS deg
            ORDER BY deg DESC
            LIMIT $max_count
            RETURN a.id AS id, deg
        """, {"label": cfg["seed_label"], "max_count": max_seeds})
        seed_ids = [r["id"] for r in records]
        print(f"  Degree-based {len(seed_ids):,} fraud seeds")

    if not seed_ids:
        print("  Không có fraud seed → anti_trustrank=0.")
        node_records = conn.run("MATCH (a:Account) RETURN a.id AS id, a.label AS label")
        df = pd.DataFrame(node_records)
        df["anti_trustrank"] = 0.0
        return df

    ppr_query = """
        MATCH (a:Account)
        WHERE a.id IN $ids
        WITH collect(a) AS fraudSeeds
        CALL gds.pageRank.write(
            $graphName,
            {
                maxIterations:  $maxIterations,
                dampingFactor:  $dampingFactor,
                tolerance:      $tolerance,
                sourceNodes:    fraudSeeds,
                writeProperty:  'anti_trustrank'
            }
        )
        YIELD nodePropertiesWritten, ranIterations, didConverge
        RETURN nodePropertiesWritten, ranIterations, didConverge
    """
    t0 = time.time()
    result = conn.run(ppr_query, {
        "graphName":     graph_name,
        "maxIterations": cfg["maxIterations"],
        "dampingFactor": cfg["dampingFactor"],
        "tolerance":     cfg["tolerance"],
        "ids":           seed_ids,
    })
    if result:
        r = result[0]
        print(f"  Ghi {r['nodePropertiesWritten']:,} nodes | hội tụ: {r['didConverge']} | {time.time()-t0:.1f}s")
        if not r['didConverge']:
            print("  ⚠ Anti-TrustRank chưa hội tụ!")

    records = conn.run("""
        MATCH (a:Account)
        WHERE a.anti_trustrank IS NOT NULL
        RETURN a.id AS id, a.anti_trustrank AS anti_trustrank, a.label AS label
        ORDER BY anti_trustrank DESC
    """)
    if not records:
        node_records = conn.run("MATCH (a:Account) RETURN a.id AS id, a.label AS label")
        df = pd.DataFrame(node_records)
        df["anti_trustrank"] = 0.0
        return df

    df = pd.DataFrame(records)
    print(f"✓ Anti-TrustRank xong. {len(df):,} nodes.")

    # Signal check
    g = df.groupby("label")["anti_trustrank"].mean()
    fraud_atr  = g.get(1, 0)
    normal_atr = g.get(0, 0)
    print(f"  Avg ATR — normal={normal_atr:.2e}, fraud={fraud_atr:.2e}")
    if fraud_atr > normal_atr:
        print("  ✓ ATR đúng chiều: fraud > normal")
    else:
        print("  ⚠ ATR sai chiều: normal >= fraud — signal yếu hoặc đảo ngược")

    return df


t0 = time.time()
df_atr = run_anti_trustrank(conn)
print(f"  Thời gian: {time.time()-t0:.1f}s")
df_atr.head()

[Anti-TrustRank] Chuẩn bị 5000 fraud seed nodes (strategy=random)...
  Random 5,000 fraud seeds từ 41,857
  Ghi 2,077,023 nodes | hội tụ: True | 15.4s
✓ Anti-TrustRank xong. 2,077,023 nodes.
  Avg ATR — normal=1.21e-03, fraud=5.24e-02
  ✓ ATR đúng chiều: fraud > normal
  Thời gian: 132.7s


,id,anti_trustrank,label
0,2130128_833585BB0,2.687491,1
1,255931_819AFC6C0,2.328289,1
2,121225_80BC80550,2.230613,1
3,22667_81D3EFE20,2.222412,1
4,240010_80F382180,2.114512,1


## Bước 9 — Auto-calibrate Weights & Tính Trust Score

In [10]:
print("→ Dùng weights mặc định từ config")
ACTIVE_WEIGHTS = TRUST_SCORE_WEIGHTS
print(f"Active weights: {ACTIVE_WEIGHTS}")

→ Dùng weights mặc định từ config
Active weights: {'trustrank_w': 0.0, 'pagerank_w': 0.2, 'anti_trustrank_w': 0.8}


In [11]:
# Suppress Neo4j deprecation warnings
import warnings
try:
    from neo4j.warnings import Neo4jWarning
    warnings.filterwarnings("ignore", category=Neo4jWarning)
except ImportError:
    pass

def _percentile_normalize(series):
    """Zero-aware rank normalization. Zeros → 0.0, non-zeros → [0,1]."""
    s_min = series.min()
    s_out = pd.Series(0.0, index=series.index)
    mask = series > s_min
    if mask.sum() > 0:
        s_out[mask] = series[mask].rank(pct=True, method='min')
    return s_out

def _hybrid_normalize(series):
    """log1p + rank. Chỉ dùng khi KHÔNG có zero-inflation (min > 0)."""
    return np.log1p(series).rank(pct=True)

def compute_trust_score(df_pagerank, df_trustrank, df_anti_trustrank, weights=None):
    weights = weights or ACTIVE_WEIGHTS

    df = df_trustrank[["id", "trustrank", "label"]].copy()
    df = df.merge(df_pagerank[["id", "pagerank"]], on="id", how="left")
    df = df.merge(df_anti_trustrank[["id", "anti_trustrank"]], on="id", how="left")
    for col in ["trustrank", "pagerank", "anti_trustrank"]:
        df[col] = df[col].fillna(0.0)

    # AntiTrustRank: cũng zero-inflated → percentile
    df["pagerank_norm"]       = _hybrid_normalize(df["pagerank"])
    df["trustrank_norm"]      = _percentile_normalize(df["trustrank"])
    df["anti_trustrank_norm"] = _percentile_normalize(df["anti_trustrank"])

    # Sanity check
    for col in ["trustrank_norm", "pagerank_norm", "anti_trustrank_norm"]:
        vmin, vmax = df[col].min(), df[col].max()
        assert 0 <= vmin and vmax <= 1.001, f"{col} out of [0,1]: [{vmin:.4f}, {vmax:.4f}]"

    # Signal direction check
    if "label" in df.columns and df["label"].notna().any():
        fraud_mask  = df["label"] == 1
        normal_mask = df["label"] == 0
        if fraud_mask.any() and normal_mask.any():
            print("\n[Signal check] normal_mean - fraud_mean:")
            for feat, expected_positive in [
                ("trustrank_norm",      True),
                ("pagerank_norm",       False),
                ("anti_trustrank_norm", False),
            ]:
                sep = df.loc[normal_mask, feat].mean() - df.loc[fraud_mask, feat].mean()
                if expected_positive:
                    ok = "✓" if sep > 0 else "✗ WRONG — TR sẽ bị exclude"
                else:
                    ok = "✓" if sep < 0 else "✗ WRONG DIRECTION"
                print(f"  {feat:25s}: sep={sep:+.4f}  {ok}")

    w_tr  = weights["trustrank_w"]
    w_pr  = weights["pagerank_w"]
    w_atr = weights["anti_trustrank_w"]
    print(f"\n[Trust Score] Weights: +{w_tr:.3f}*TR  -{w_pr:.3f}*PR  -{w_atr:.3f}*ATR")

    df["trust_score_raw"] = (
        + w_tr  * df["trustrank_norm"]
        - w_pr  * df["pagerank_norm"]
        - w_atr * df["anti_trustrank_norm"]
    )

    ts = df["trust_score_raw"]
    ts_min, ts_max = ts.min(), ts.max()
    df["trust_score"] = (ts - ts_min) / (ts_max - ts_min) if ts_max > ts_min else 0.5

    df["credit_score"] = (
        CREDIT_SCORE_MIN + df["trust_score"] * (CREDIT_SCORE_MAX - CREDIT_SCORE_MIN)
    ).round(0).astype(int)

    p25 = df["trust_score"].quantile(0.25)
    p50 = df["trust_score"].quantile(0.50)
    p75 = df["trust_score"].quantile(0.75)

    conditions = [df["trust_score"] >= p75, df["trust_score"] >= p50, df["trust_score"] >= p25]
    df["trust_level"]  = np.select(conditions, ["HIGH","MEDIUM","LOW"], default="VERY_LOW")
    df["credit_level"] = df["credit_score"].apply(
        lambda cs: "Excellent" if cs>=750 else "Good" if cs>=650 else
                   "Fair" if cs>=550 else "Poor" if cs>=450 else "Very Poor"
    )
    df = df.sort_values("credit_score", ascending=False).reset_index(drop=True)

    print("\n" + "="*60)
    if "label" in df.columns:
        fraud  = df[df["label"] == 1]
        normal = df[df["label"] == 0]
        if not fraud.empty and not normal.empty:
            sep = normal["trust_score"].mean() - fraud["trust_score"].mean()
            print(f"Separation (normal-fraud): {sep:+.4f} {'✓ GOOD' if sep>0.05 else '~ OK' if sep>0 else '✗ BAD'}")
    return df


t0 = time.time()
df_final = compute_trust_score(
    df_pagerank=df_pr,
    df_trustrank=df_tr,
    df_anti_trustrank=df_atr,
)
print(f"  Thời gian: {time.time()-t0:.1f}s")
df_final.head(10)


[Signal check] normal_mean - fraud_mean:
  trustrank_norm           : sep=-0.0108  ✗ WRONG — TR sẽ bị exclude
  pagerank_norm            : sep=-0.0601  ✓
  anti_trustrank_norm      : sep=-0.2720  ✓

[Trust Score] Weights: +0.000*TR  -0.200*PR  -0.800*ATR

Separation (normal-fraud): +0.2340 ✓ GOOD
  Thời gian: 9.7s


,id,trustrank,label,pagerank,anti_trustrank,pagerank_norm,trustrank_norm,anti_trustrank_norm,trust_score_raw,trust_score,credit_score,trust_level,credit_level
0,380710_81FEA53E0,0.0,0,0.15,0.0,0.093183,0.0,0.0,-0.018637,1.0,900,HIGH,Excellent
1,380709_81E53BC50,0.0,0,0.15,0.0,0.093183,0.0,0.0,-0.018637,1.0,900,HIGH,Excellent
2,380709_81D79D8D0,0.0,0,0.15,0.0,0.093183,0.0,0.0,-0.018637,1.0,900,HIGH,Excellent
3,380703_821E02C60,0.0,0,0.15,0.0,0.093183,0.0,0.0,-0.018637,1.0,900,HIGH,Excellent
4,380702_81F9BAF80,0.0,0,0.15,0.0,0.093183,0.0,0.0,-0.018637,1.0,900,HIGH,Excellent
5,220538_825F04680,0.0,0,0.15,0.0,0.093183,0.0,0.0,-0.018637,1.0,900,HIGH,Excellent
6,187336_845ABBED0,0.0,0,0.15,0.0,0.093183,0.0,0.0,-0.018637,1.0,900,HIGH,Excellent
7,380769_81E96A690,0.0,0,0.15,0.0,0.093183,0.0,0.0,-0.018637,1.0,900,HIGH,Excellent
8,380768_81D939F30,0.0,0,0.15,0.0,0.093183,0.0,0.0,-0.018637,1.0,900,HIGH,Excellent
9,380768_81D7E21D0,0.0,0,0.15,0.0,0.093183,0.0,0.0,-0.018637,1.0,900,HIGH,Excellent


## Bước 10 — Debug & Phân tích phân phối score

In [12]:
# Percentile table
percentiles = [5, 10, 25, 50, 75, 90, 95, 99]
print("\nTrust Score Percentiles:")
for p in percentiles:
    v = df_final["trust_score"].quantile(p / 100)
    print(f"  P{p:02d}: {v:.4f}")

# Fraud vs Normal breakdown
if "label" in df_final.columns and df_final["label"].notna().any():
    fraud  = df_final[df_final["label"] == 1]
    normal = df_final[df_final["label"] == 0]
    print(f"\nFraud  (n={len(fraud):,}): avg_trust={fraud['trust_score'].mean():.4f}, avg_credit={fraud['credit_score'].mean():.0f}")
    print(f"Normal (n={len(normal):,}): avg_trust={normal['trust_score'].mean():.4f}, avg_credit={normal['credit_score'].mean():.0f}")
    sep = normal["trust_score"].mean() - fraud["trust_score"].mean()
    print(f"Separation: {sep:+.4f} {'✓ GOOD' if sep > 0.05 else '~ OK' if sep > 0 else '✗ BAD'}")


Trust Score Percentiles:
  P05: 0.2332
  P10: 0.4077
  P25: 0.8288
  P50: 0.8803
  P75: 0.9648
  P90: 1.0000
  P95: 1.0000
  P99: 1.0000

Fraud  (n=41,857): avg_trust=0.5889, avg_credit=653
Normal (n=2,035,166): avg_trust=0.8228, avg_credit=794
Separation: +0.2340 ✓ GOOD


## Bước 11 — Validation

In [13]:
from sklearn.metrics import roc_auc_score

def validate_trust_scores(df: pd.DataFrame):
    print("\n" + "="*60)
    print("VALIDATION REPORT")
    print("="*60)

    # Test 1: Range
    assert df["trust_score"].between(0, 1).all(),      "FAIL: trust_score ngoài [0,1]!"
    assert df["credit_score"].between(300, 900).all(), "FAIL: credit_score ngoài [300,900]!"
    print("✓ trust_score ∈ [0, 1]")
    print("✓ credit_score ∈ [300, 900]")

    # Test 2: Distribution
    desc = df["trust_score"].describe()
    print(f"\nPhân bố trust_score:")
    print(f"  Min={desc['min']:.4f} | P25={desc['25%']:.4f} | Median={desc['50%']:.4f} | P75={desc['75%']:.4f} | Max={desc['max']:.4f} | Std={desc['std']:.4f}")

    if desc["50%"] < 0.15:
        print("  ✗ CẢNH BÁO: Median < 0.15 → phân phối skewed!")
    elif desc["50%"] < 0.40:
        print("  ~ Median hơi thấp nhưng chấp nhận được.")
    else:
        print(f"  ✓ Median OK ({desc['50%']:.4f})")

    if desc["std"] < 0.05:
        print("  ✗ CẢNH BÁO: Std < 0.05 → phân phối quá tập trung!")
    else:
        print(f"  ✓ Std={desc['std']:.4f} (đủ phân tán)")

    # [AUDIT-FIX9] Zero-inflation check
    zero_pct = (df["trust_score"] == 0).sum() / len(df) * 100
    if zero_pct > 10:
        print(f"  ⚠ {zero_pct:.1f}% accounts có trust_score=0 — zero-inflation!")
    else:
        print(f"  ✓ Zero-inflation OK ({zero_pct:.1f}% = 0)")

    # Test 3: Trust level distribution
    print("\nPhân bổ trust_level:")
    level_counts = df["trust_level"].value_counts()
    total = len(df)
    very_low_pct = level_counts.get("VERY_LOW", 0) / total * 100
    for level in ["HIGH", "MEDIUM", "LOW", "VERY_LOW"]:
        n   = level_counts.get(level, 0)
        pct = n / total * 100
        sym = "✓" if pct > 10 else "~"
        print(f"  {sym} {level:<10}: {n:>8,} ({pct:5.1f}%)")
    if very_low_pct > 70:
        print(f"  ✗ CẢNH BÁO: {very_low_pct:.1f}% VERY_LOW → quá nhiều!")

    # Test 4: Fraud vs Normal separation + AUC
    if "label" in df.columns:
        fraud_df  = df[df["label"] == 1]
        normal_df = df[df["label"] == 0]
        if not fraud_df.empty and not normal_df.empty:
            sep = normal_df["trust_score"].mean() - fraud_df["trust_score"].mean()
            print(f"\nFraud vs Normal separation: {sep:+.4f} {'✓' if sep > 0.05 else '~' if sep > 0 else '✗'}")

            try:
                # trust_score cao = tin cậy → (1-trust_score) = fraud risk
                auc = roc_auc_score(df["label"].values, 1 - df["trust_score"].values)
                print(f"AUC-ROC: {auc:.4f} {'✓ tốt' if auc >= 0.70 else '~ trung bình' if auc >= 0.60 else '✗ thấp'}")

                if auc < 0.5:
                    auc_direct = roc_auc_score(df["label"].values, df["trust_score"].values)
                    print(f"  🚨 AUC < 0.5 — trust_score VẪN cao hơn cho fraud!")
                    print(f"  AUC(trust_score trực tiếp) = {auc_direct:.4f}")
                    print(f"  → Kiểm tra lại signal direction và weights")
                else:
                    print(f"  ✓ Đúng chiều: fraud trust thấp, normal trust cao.")
            except Exception as e:
                print(f"Không tính AUC: {e}")

    # Test 5: Outlier
    ts = df["trust_score"]
    q1, q3 = ts.quantile(0.25), ts.quantile(0.75)
    iqr = q3 - q1
    outliers = df[(ts < q1 - 3*iqr) | (ts > q3 + 3*iqr)]
    print(f"\nOutliers (3×IQR): {len(outliers)} ({len(outliers)/total*100:.2f}%)")

    print("\n" + "="*60)
    print("VALIDATION HOÀN THÀNH")
    print("="*60)


validate_trust_scores(df_final)


VALIDATION REPORT
✓ trust_score ∈ [0, 1]
✓ credit_score ∈ [300, 900]

Phân bố trust_score:
  Min=0.0000 | P25=0.8288 | Median=0.8803 | P75=0.9648 | Max=1.0000 | Std=0.2316
  ✓ Median OK (0.8803)
  ✓ Std=0.2316 (đủ phân tán)
  ✓ Zero-inflation OK (0.0% = 0)

Phân bổ trust_level:
  ✓ HIGH      :  519,256 ( 25.0%)
  ✓ MEDIUM    :  762,503 ( 36.7%)
  ✓ LOW       :  276,008 ( 13.3%)
  ✓ VERY_LOW  :  519,256 ( 25.0%)

Fraud vs Normal separation: +0.2340 ✓
AUC-ROC: 0.7107 ✓ tốt
  ✓ Đúng chiều: fraud trust thấp, normal trust cao.

Outliers (3×IQR): 215690 (10.38%)

VALIDATION HOÀN THÀNH


In [ ]:
!./neo4j-community-2026.03.1/bin/neo4j-admin database dump neo4j --to-path=/kaggle/working/